# 🧠 PlantPulse AI — Model Training and Serialization
## Task: Binary Failure Prediction & Multi-Class Failure Type Classification

---

This notebook covers the machine-learning pipeline for **PlantPulse AI**:
1. Feature engineering and preprocessing (handling categorical variables, feature scaling).
2. Dealing with severe class imbalance using SMOTE (Synthetic Minority Over-sampling Technique).
3. Training and evaluating binary classifiers (Logistic Regression vs. XGBoost) to predict *if* a machine will fail.
4. Training a multi-class classifier on failed instances to diagnose *why* it failed (HDF, OSF, PWF, TWF, or Other).
5. Serializing the final preprocessor and model artifacts using `joblib` for deployment.

---

In [1]:
# ── Core imports ───────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
from pathlib import Path
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

sns.set_theme(style='darkgrid')
print('Libraries loaded successfully ✓')

Libraries loaded successfully ✓


---
## 1 · Load the Dataset

We load the dataset from the top-level `data/` directory.

In [2]:
# ── Load Data ─────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
DATA_PATH    = NOTEBOOK_DIR / '../../data/ai4i2020.csv'
assert DATA_PATH.exists(), f'Dataset not found at: {DATA_PATH.resolve()}'

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

Dataset shape: 10,000 rows × 14 columns


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


---
## 2 · Feature Preparation & Preprocessing Pipeline

To prepare our dataset for modelling:
- We **drop identifiers** such as `UDI` (unique row index) and `Product ID` (specific machine serial number) since they contain no generalisable predictive information.
- We **encode `Type`** (the machine quality level: L, M, H) using One-Hot Encoding to convert it into numeric features without assuming any mathematical ordering.
- We **scale numeric sensor features** using standardisation (mean = 0, variance = 1) to ensure features with larger magnitudes (like Rotational speed ~1400rpm) do not dominate features with smaller scales (like Torque ~40Nm or Tool Wear ~100min), which is particularly important for models like Logistic Regression.

We define a scikit-learn `ColumnTransformer` to automate this preprocessing in a repeatable, production-ready pipeline.

In [3]:
# Define features (X) and binary target (y)
X = df[['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']]
y = df['Machine failure']

# Set up the Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), ['Type']),
        ('num', StandardScaler(), ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'])
    ])

# Split into Train & Test (80/20 ratio), stratifying on the target to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit preprocessor on X_train only and transform both splits
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f'Train set preprocessed shape: {X_train_proc.shape}')
print(f'Test set preprocessed shape: {X_test_proc.shape}')

Train set preprocessed shape: (8000, 7)
Test set preprocessed shape: (2000, 7)


---
## 3 · Handling Class Imbalance with SMOTE

### ⚠️ Crucial Rule: SMOTE Only on Training Data
SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples of the minority class (machine failures) to balance the class distribution.

We must **unconditionally apply SMOTE only to the training split** and **never to the test split**:
1. **Data Leakage Prevention**: If we apply SMOTE to the entire dataset before splitting, the synthetic test samples will be constructed using information from the training samples. This artificially inflates performance metrics.
2. **Real-world Generalisation**: The test set must represent the actual distribution the model will encounter in production, where failures are rare (approx. 3.4%). Over-sampling the test set would corrupt our evaluation metrics, giving us a false sense of security.


In [4]:
# Apply SMOTE to training split only
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)

print('── Class Distribution Before SMOTE ──')
print(y_train.value_counts())
print('\n── Class Distribution After SMOTE ──')
print(y_train_res.value_counts())

── Class Distribution Before SMOTE ──
Machine failure
0    7729
1     271
Name: count, dtype: int64

── Class Distribution After SMOTE ──
Machine failure
0    7729
1    7729
Name: count, dtype: int64


---
## 4 · Training Binary Classifiers: Logistic Regression vs. XGBoost

We train two classifiers for comparison:
1. **Logistic Regression** — A simple, linear baseline model that is highly interpretable.
2. **XGBoost Classifier** — A state-of-the-art tree-boosting algorithm capable of capturing complex non-linear interactions between sensor features (e.g., speed vs. torque).

In [5]:
# Train Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_res, y_train_res)
y_pred_lr = lr_model.predict(X_test_proc)

# Train XGBoost Classifier
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_model.predict(X_test_proc)

print('Both models trained successfully ✓')

Both models trained successfully ✓


---
## 5 · Binary Classifier Evaluation

### 📊 Why Accuracy Alone is Misleading
In imbalanced datasets, accuracy is a highly deceptive metric. If a dataset has 96.6% non-failures and 3.4% failures, a dummy model that simply predicts "No Failure" all the time will achieve a stellar **96.6% accuracy**, while failing to catch a single actual failure (0% Recall).

For predictive maintenance, we care about:
- **Recall (Sensitivity)**: What percentage of actual failures did we catch? (Minimises unexpected downtime).
- **Precision**: When the model predicts a failure, is it correct? (Minimises false alarms and wasted maintenance visits).
- **F1-Score**: The harmonic mean of Precision and Recall, which gives a balanced indicator of model performance on the minority class.


In [6]:
def print_evaluation(name, y_true, y_pred):
    print(f'=== {name} ===')
    print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precision: {precision_score(y_true, y_pred):.4f}')
    print(f'Recall   : {recall_score(y_true, y_pred):.4f}')
    print(f'F1-Score : {f1_score(y_true, y_pred):.4f}')
    print('\nConfusion Matrix:')
    print(confusion_matrix(y_true, y_pred))
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred))
    print('=' * 40 + '\n')

print_evaluation('Logistic Regression Baseline', y_test, y_pred_lr)
print_evaluation('XGBoost Classifier', y_test, y_pred_xgb)

=== Logistic Regression Baseline ===
Accuracy : 0.8290
Precision: 0.1451
Recall   : 0.8235
F1-Score : 0.2467

Confusion Matrix:
[[1602  330]
 [  12   56]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.83      0.90      1932
           1       0.15      0.82      0.25        68

    accuracy                           0.83      2000
   macro avg       0.57      0.83      0.58      2000
weighted avg       0.96      0.83      0.88      2000


=== XGBoost Classifier ===
Accuracy : 0.9800
Precision: 0.6707
Recall   : 0.8088
F1-Score : 0.7333

Confusion Matrix:
[[1905   27]
 [  13   55]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.67      0.81      0.73        68

    accuracy                           0.98      2000
   macro avg       0.83      0.90      0.86      2000
weighted avg       0.98      0.98      0.98      200

---
## 6 · Failure Type Classification (Multi-class Model)

If a machine is predicted to fail, we want to diagnose the root cause so technicians know what to inspect.
We train a separate multi-class classifier on instances where `Machine failure == 1`.

### Failure Mapping
- We check the five binary failure flags (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`).
- If a machine failed but none of these flags are active, or if it had a rare Random Failure (`RNF`), we categorize it as `Other` to maintain robust classification boundaries.


In [7]:
# Filter dataset to failed machines only
failed_df = df[df['Machine failure'] == 1].copy()

def get_failure_type(row):
    for f in ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']:
        if row[f] == 1:
            return f
    return 'Other'

failed_df['Failure_Type'] = failed_df.apply(get_failure_type, axis=1)

# Features and Target for multi-class
X_fail = failed_df[['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']]
y_fail = failed_df['Failure_Type']

# Preprocess features using the preprocessor fitted on the full dataset
X_fail_proc = preprocessor.transform(X_fail)

# Encode the target labels
le = LabelEncoder()
y_fail_enc = le.fit_transform(y_fail)

# Train/Test Split (80/20) on failure-only instances
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fail_proc, y_fail_enc, test_size=0.2, random_state=42, stratify=y_fail_enc
)

# Train XGBoost Multi-Class Classifier
xgb_fail_type_model = XGBClassifier(random_state=42, eval_metric='mlogloss')
xgb_fail_type_model.fit(X_train_f, y_train_f)
y_pred_f = xgb_fail_type_model.predict(X_test_f)

print('=== Multi-class Failure Type Classification (XGBoost) ===')
print(f'Accuracy: {accuracy_score(y_test_f, y_pred_f):.4f}')
print('\nClassification Report:')
print(classification_report(y_test_f, y_pred_f, target_names=le.classes_))

=== Multi-class Failure Type Classification (XGBoost) ===
Accuracy: 0.8676

Classification Report:
              precision    recall  f1-score   support

         HDF       0.85      1.00      0.92        23
         OSF       0.77      0.62      0.69        16
       Other       1.00      1.00      1.00         2
         PWF       0.95      1.00      0.97        18
         TWF       0.86      0.67      0.75         9

    accuracy                           0.87        68
   macro avg       0.89      0.86      0.87        68
weighted avg       0.86      0.87      0.86        68



---
## 7 · Model Serialization

We save our preprocessing pipeline and trained models using `joblib`. 
For the multi-class classifier, we save the model alongside its mapped classes so they can be decoded back to labels in production.

In [8]:
# Ensure models directory exists
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

# Save Preprocessor
joblib.dump(preprocessor, models_dir / 'preprocessor.pkl')

# Save Binary Classifier (XGBoost)
joblib.dump(xgb_model, models_dir / 'failure_model.pkl')

# Save Multi-Class Classifier and Class Mapping
joblib.dump({
    'model': xgb_fail_type_model,
    'classes': le.classes_
}, models_dir / 'failure_type_model.pkl')

print('Models serialized successfully inside ml-service/models/ ✓')
print(f'- preprocessor.pkl       : {os.path.getsize(models_dir / "preprocessor.pkl")/1024:.2f} KB')
print(f'- failure_model.pkl      : {os.path.getsize(models_dir / "failure_model.pkl")/1024:.2f} KB')
print(f'- failure_type_model.pkl : {os.path.getsize(models_dir / "failure_type_model.pkl")/1024:.2f} KB')

Models serialized successfully inside ml-service/models/ ✓
- preprocessor.pkl       : 3.05 KB
- failure_model.pkl      : 257.65 KB
- failure_type_model.pkl : 425.13 KB


---
## 8 · Summary of Achieved Metrics

These metrics are the real, non-placeholder performance figures on the test sets.

In [9]:
print('=' * 50)
print('             FINAL PERFORMANCE METRICS')
print('=' * 50)
print('🛡️  BINARY FAILURE PREDICTION (XGBoost)')
print(f'  Accuracy  : {accuracy_score(y_test, y_pred_xgb) * 100:.2f}%')
print(f'  Precision : {precision_score(y_test, y_pred_xgb) * 100:.2f}%')
print(f'  Recall    : {recall_score(y_test, y_pred_xgb) * 100:.2f}%')
print(f'  F1-Score  : {f1_score(y_test, y_pred_xgb) * 100:.2f}%')
print()
print('🔬  MULTI-CLASS FAILURE ROOT CAUSE (XGBoost)')
print(f'  Accuracy  : {accuracy_score(y_test_f, y_pred_f) * 100:.2f}%')
print('=' * 50)

             FINAL PERFORMANCE METRICS
🛡️  BINARY FAILURE PREDICTION (XGBoost)
  Accuracy  : 98.00%
  Precision : 67.07%
  Recall    : 80.88%
  F1-Score  : 73.33%

🔬  MULTI-CLASS FAILURE ROOT CAUSE (XGBoost)
  Accuracy  : 86.76%
